In [ ]:
# Run this setup cell first in Google Colab. No API key is needed.
%pip install -q langgraph


In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str


def classify(state: TicketState) -> dict:
    # Billing keywords take priority; other tickets go to technical support.
    billing_keywords = ('invoice', 'payment', 'refund', 'charged', 'billing')
    ticket = state['ticket'].lower()
    category = 'billing' if any(word in ticket for word in billing_keywords) else 'technical'
    return {'category': category}


def route_ticket(state: TicketState) -> Literal['billing', 'technical']:
    return 'billing' if state['category'] == 'billing' else 'technical'


def billing(state: TicketState) -> dict:
    return {'reply': 'Our billing team can help. Please provide your invoice number.'}


def technical(state: TicketState) -> dict:
    return {'reply': 'Our technical team can help. Please share the error message and steps to reproduce it.'}


builder = StateGraph(TicketState)
builder.add_node('classify', classify)
builder.add_node('billing', billing)
builder.add_node('technical', technical)
builder.add_edge(START, 'classify')
builder.add_conditional_edges(
    'classify',
    route_ticket,
    {'billing': 'billing', 'technical': 'technical'},
)
builder.add_edge('billing', END)
builder.add_edge('technical', END)
graph = builder.compile()


In [ ]:
# Run one sample ticket through each route.
sample_tickets = [
    'I was charged twice for my latest invoice.',
    'The app crashes whenever I try to sign in.',
]

for ticket in sample_tickets:
    result = graph.invoke({'ticket': ticket, 'category': '', 'reply': ''})
    print('Ticket:', result['ticket'])
    print('Category:', result['category'])
    print('Reply:', result['reply'])
    print()


In [ ]:
from IPython.display import Image, display

# Render the graph using the online Mermaid renderer; internet access is required.
# Only the graph structure is sent to the renderer, not the sample tickets.
display(Image(graph.get_graph().draw_mermaid_png()))
